In [1]:
from google import genai
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

In [4]:
def moderate_input(complaint):

    prompt = f"""
    You are a safety moderation system for an e-commerce
    customer support assistant.

    Analyze the following customer input:

    {complaint}

    Classify the input as either:

    SAFE
    or
    UNSAFE

    UNSAFE includes:
    - Threats of violence
    - Instructions for harmful or illegal activity
    - Explicit sexual content
    - Hate or harassment targeting protected groups
    - Dangerous activities

    Normal customer complaints, frustration, refund requests,
    product complaints, and negative feedback are SAFE.

    Return exactly:

    Status: <SAFE or UNSAFE>
    Reason: <brief reason>
    """

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text

In [5]:
def classify_complaint(complaint):

    prompt = f"""
    You are an e-commerce customer complaint classification system.

    Analyze the following complaint:

    {complaint}

    Choose exactly one category:

    - Order Issue
    - Delivery Issue
    - Product Issue
    - Return/Replacement
    - Refund Issue
    - Payment Issue
    - Account Issue
    - Cancellation
    - General Inquiry

    Choose exactly one priority:

    - Low
    - Medium
    - High

    Return exactly:

    Category: <category>
    Priority: <priority>
    """

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text

In [6]:
def process_complaint(complaint):

    prompt = f"""
    You are an e-commerce customer support input processing system.

    Analyze the following customer complaint:

    {complaint}

    Extract:

    1. Order ID
    2. Product
    3. Issue
    4. Customer Request
    5. Date

    Rules:
    - Extract information only if it is present.
    - Do not invent or assume information.
    - If information is missing, write "Not provided".

    Return exactly:

    Order ID: <value>
    Product: <value>
    Issue: <value>
    Customer Request: <value>
    Date: <value>
    """

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text

In [16]:
def determine_resolution(
    complaint,
    classification,
    processed_information
):

    prompt = f"""
    You are an e-commerce customer support resolution system.

    Customer Complaint:
    {complaint}

    Complaint Classification:
    {classification}

    Extracted Information:
    {processed_information}

    Determine the appropriate resolution.

    Rules:
    - Do not assume that the customer is eligible for a return,replacement, refund, or exchange.
    - Do not interpret a date such as "last week" as proof of policy eligibility.
    - If company policy has not been provided, state that policy verification
      is required before confirming eligibility.
        Return exactly:

    Issue: <main issue>
    Resolution: <recommended resolution>
    Reason: <brief reason>
    """

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text

In [17]:
required_fields = [
    "Issue",
    "Resolution",
    "Reason"
]

def validate_resolution_output(result):

    lines = result.splitlines()

    extracted_fields = {}

    for line in lines:

        for field in required_fields:

            if line.startswith(field + ":"):

                value = line.split(":", 1)[1].strip()

                extracted_fields[field] = value

    missing_fields = [
        field
        for field in required_fields
        if field not in extracted_fields
    ]

    empty_fields = [
        field
        for field, value in extracted_fields.items()
        if not value
    ]

    return {
        "extracted_fields": extracted_fields,
        "missing_fields": missing_fields,
        "empty_fields": empty_fields,
        "valid": len(missing_fields) == 0 and len(empty_fields) == 0
    }

In [18]:
def check_output(complaint, resolution):

    prompt = f"""
    You are an output quality checker for an e-commerce
    customer complaint resolution system.

    Customer Complaint:
    {complaint}

    Generated Resolution:
    {resolution}

    Check whether the resolution:

    1. Addresses the customer's actual complaint.
    2. Uses only information available from the complaint.
    3. Avoids inventing order details.
    4. Avoids unsupported promises.
    5. Provides a reasonable next step.

    Return exactly:

    Status: <PASS or FAIL>
    Reason: <brief reason>
    """

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text

In [19]:
def complaint_resolution_system(complaint):

    print("\n[1/5] Moderating complaint...")

    moderation = moderate_input(complaint)

    print("Moderation completed.")
    print(moderation)

    if "Status: UNSAFE" in moderation:
        return {
            "status": "BLOCKED",
            "moderation": moderation
        }

    print("\n[2/5] Classifying complaint...")

    classification = classify_complaint(complaint)

    print("Classification completed.")
    print(classification)

    print("\n[3/5] Processing complaint information...")

    processed_information = process_complaint(complaint)

    print("Input processing completed.")
    print(processed_information)

    print("\n[4/5] Determining resolution...")

    resolution = determine_resolution(
        complaint,
        classification,
        processed_information
    )

    print("Resolution generated.")
    print(resolution)

    print("\n[5/5] Checking output...")

    validation = validate_resolution_output(resolution)

    if validation["valid"]:

        quality_check = check_output(complaint,resolution)

    else:

        quality_check = "Status: FAIL\nReason: Invalid output structure."

    print("Output checking completed.")

    return {
        "status": "COMPLETED",
        "moderation": moderation,
        "classification": classification,
        "processed_information": processed_information,
        "resolution": resolution,
        "validation": validation,
        "quality_check": quality_check
    }

In [20]:
complaint = input("Enter your complaint: ")

result = complaint_resolution_system(complaint)

Enter your complaint:   I ordered a black Nike shoe last week. The order ID is #12345, but I received the wrong size. I want to exchange it.



[1/5] Moderating complaint...
Moderation completed.
Status: SAFE
Reason: The customer is making a standard product exchange request due to receiving the wrong size, which is a normal customer support inquiry.

[2/5] Classifying complaint...
Classification completed.
Category: Return/Replacement
Priority: Medium

[3/5] Processing complaint information...
Input processing completed.
Order ID: #12345
Product: Black Nike shoe
Issue: Received the wrong size
Customer Request: Exchange it
Date: Last week

[4/5] Determining resolution...
Resolution generated.
Issue: Received the wrong size for order #12345 (Black Nike shoe) and requested an exchange.
Resolution: Initiate the exchange process pending policy verification.
Reason: Policy verification is required before confirming eligibility as return/exchange policies and timeframes have not been provided, and "last week" cannot be assumed as proof of eligibility.

[5/5] Checking output...
Output checking completed.
